In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
#from scipy import interpolate
#from scipy.spatial import cKDTree
import coordinateSystems as cs
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import glob
import interpolation
import matplotlib.path as mpth
from cartopy.io.shapereader import Reader
import json
import geojson
import rasterio
#import rioxarray as rxr
import geopandas as gpd
from shapely.geometry import mapping
#from metpy.units import units
#import metpy.calc as mpcalc
import pandas as pd
import os

In [ ]:
def get_basin_points(lake_csv):
    df = pd.read_csv(lake_csv, usecols=['long','lat'])
    return df[['long', 'lat']].to_numpy(dtype=float)

In [ ]:
def mask_inside_polygon_path(lon,lat,data,datadim=2,basin_only=False):
    '''
    lon,lat must be 2D mesh of coordinates
    '''
    dirlist = os.getcwd().split('/')[:-1]
    mydir=''
    for d in dirlist:
        mydir = mydir+d+'/'
#     print(mydir)
    if 'Abby.Hutson' in mydir:
        mydir='/home/Abby.Hutson/'
    # import datafile containing lon/lat points of basin polygons
    base_outline_fn = '_lbrm_outline.csv'
    ont_points = get_basin_points(f'ON{base_outline_fn}')
    erie_points= get_basin_points(f'ERI{base_outline_fn}')
    hur_points = get_basin_points(f'HU{base_outline_fn}')
    mich_points= get_basin_points(f'MIC{base_outline_fn}')
    sup_points = get_basin_points(f'SUP{base_outline_fn}')
    basin_points=ds.basin.data

    path      = mpth.Path(basin_points) # This is the entire basin, not just individual lakes
    basin_mask= path.contains_points(np.hstack((lon.flatten()[:,np.newaxis],lat.flatten()[:,np.newaxis])),
                                     radius=-0.1)
    rain_basin=basin_data(basin_mask,data,datadim)
    if basin_only is True:
        return rain_basin
    else:
        # Find locations that fall within the basin polygons
        path     = mpth.Path(mich_points) #<--create path using lat/lon data of polygon outline
        mich_mask= path.contains_points(np.hstack((lon.flatten()[:,np.newaxis],lat.flatten()[:,np.newaxis])),
                                        radius=-0.1)#<--find wrf points that fall within polygon
        path     = mpth.Path(ont_points)
        ont_mask = path.contains_points(np.hstack((lon.flatten()[:,np.newaxis],lat.flatten()[:,np.newaxis])),
                                        radius=-0.1)
        path     = mpth.Path(erie_points)
        erie_mask= path.contains_points(np.hstack((lon.flatten()[:,np.newaxis],lat.flatten()[:,np.newaxis])),
                                        radius=-0.1)
        path     = mpth.Path(hur_points)
        hur_mask = path.contains_points(np.hstack((lon.flatten()[:,np.newaxis],lat.flatten()[:,np.newaxis])),
                                        radius=-0.1)
        path     = mpth.Path(sup_points)
        sup_mask = path.contains_points(np.hstack((lon.flatten()[:,np.newaxis],lat.flatten()[:,np.newaxis])),
                                        radius=-0.1)

        # mask data within basins, set nan outside of the basins, append to year arrays
        rain_ont = basin_data(ont_mask,data)
        rain_erie= basin_data(erie_mask,data)
        rain_hur = basin_data(hur_mask,data)
        rain_mich= basin_data(mich_mask,data)
        rain_sup = basin_data(sup_mask,data)
        return rain_basin,rain_ont,rain_erie,rain_hur,rain_mich,rain_sup


In [ ]:
def get_masked_prec_lake(ds, index, gpd_shapefile,varname='tp'):
    # splitting shapefile into individual polygons is more difficult than it needs to be

    # 0: erie, 1: huron, 2: michigan, 3: ontario, 4: superior (alphabetical)
    geometry_s = gpd_shapefile.geometry.apply(mapping)[index] # grab polygon for individual lake
    geometry   = json.dumps(geometry_s) # change single quotes to double quotes (dumb, right??)
    cropping_geoms = [geojson.loads(str(geometry))] # create geojson polygon object
    era_lake       =ds.rio.clip(geometries=cropping_geoms,
                                crs=gpd_shapefile.crs,
                                all_touched=True, drop=False) #clip based on geojson polygon

    # data is organized (month, hour, lat, lon)
#     tp_lake = np.sum(era_lake[varname].data,axis=1) THIS WAS ONLY SPECIFIC TO ERA DATA
    tp_lake = era_lake[varname].data
    return tp_lake

def mask_using_shapefile_polygon(dataset,lons,lats,varname='tp',lon_vname='lon',lat_vname='lat',whole_ds=False):
    '''
    This method can only be used when lons, lats are 1D arrays
    (i.e., when the grid is defined by regular lon/lat points
    as opposed to regular earth distance points)
    '''
    # get file directory correct based on file system
    dirlist = os.getcwd().split('/')[:-1]
    mydir=''
    for d in dirlist:
        mydir = mydir+d+'/'
    if 'Abby.Hutson' in mydir:
        mydir='/home/Abby.Hutson/'

    # print(np.nanmax(lons), np.nanmax(lats), np.nanmin(lons),np.nanmin(lats))
    lats_t,lons_t = np.meshgrid(lons,lats)
    lat, lon      = lats_t.T, lons_t.T

    shapefile = f'{mydir}GreatLakesandWa/greatlakes_subbasins/greatlakes_subbasins.shp'
    reader = Reader(shapefile)

    basins  = reader.records()
    lk_erie = [basin for basin in reader.records() if basin.attributes['merge']=='lk_erie'][0]
    lk_huron= [basin for basin in reader.records() if basin.attributes['merge']=='lk_huron'][0]
    lk_mich = [basin for basin in reader.records() if basin.attributes['merge']=='lk_mich'][0]
    lk_ont  = [basin for basin in reader.records() if basin.attributes['merge']=='lk_ont'][0]
    lk_sup  = [basin for basin in reader.records() if basin.attributes['merge']=='lk_sup'][0]

    # print(np.nanmax(dataset.t.data))
    with rasterio.Env(OGR_ENABLE_PARTIAL_REPROJECTION=True) as env:
        dataset.rio.set_spatial_dims(x_dim=lon_vname,y_dim=lat_vname,inplace=True)
        dataset.rio.write_crs('epsg:4326',inplace=True)
        basin   = gpd.read_file(shapefile, crs='epsg:4326')
        ds_basin=dataset.rio.clip(basin.geometry.apply(mapping), basin.crs,
                                      all_touched=True, drop=False)
    if whole_ds is True:
        # print(np.nanmax(ds_basin.t.data))
        return ds_basin
    else:
        rainbasin=ds_basin[varname].data

        rainerie= get_masked_prec_lake(dataset, 0,basin,varname=varname)
        rainhur = get_masked_prec_lake(dataset, 1,basin,varname=varname)
        rainmich= get_masked_prec_lake(dataset, 2,basin,varname=varname)
        rainont = get_masked_prec_lake(dataset, 3,basin,varname=varname)
        rainsup = get_masked_prec_lake(dataset, 4,basin,varname=varname)

        return rainbasin,rainont,rainerie,rainhur,rainmich,rainsup